# 03: Ask the Coach

An interactive playground for the P0-6 answer generator (`rag_helper.py`):
retrieval is the vector approach that won P0-5, generation is Haiku 4.5.
Edit the `question` cell and re-run it.

In [ ]:
from dotenv import load_dotenv

from carryia.pipeline.ingest import load_corpus, build_vector_index
from carryia.serve.llm_backend import make_client, model_id
from carryia.serve.rag_helper import RAGBase, COACH_BLUNT, COACH_WHY

load_dotenv()
client = make_client()   # anthropic or bedrock, per CARRYIA_LLM_BACKEND
model = model_id()       # matching model id for that backend

In [2]:
documents = load_corpus()
index = build_vector_index(documents)   # local fastembed; first run downloads the model
len(documents)

/Users/mattheworga/Documents/dev/Carryia/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


709

In [3]:
coach = RAGBase(index=index, llm_client=client, model=model)   # instructions=COACH_WHY by default

## Ask

Change the question and re-run this cell.

In [4]:
question = "I keep getting caught in bot"

print(coach.rag(question))

# Getting Caught in Bot Lane

Based on your situation, here are the concrete actions to take:

**1. Check the map before automatically returning to bot with your ADC**
- **Action:** After any play (kill, objective, back), pause and scan where the next fight is likely to happen before heading back to lane.
- **Why it works:** Reflexively returning to bot puts you on the wrong side of the map right as the next fight breaks out elsewhere, making you an easy target. (skill-capped)

**2. Place defensive Control Wards behind your team when pushing bot**
- **Action:** When you're sieging bot lane, put a Control Ward in the bushes *behind* your team's position.
- **Why it works:** This denies the enemy teleport plays and flank setups while you're committed to the push, protecting you from getting caught from the side. (mobalytics)

**3. Recognize when to break standard bot-lane commitment**
- **Action:** Develop the judgment to know when giving up bot lane entirely is the right call—don't feel

## What did it retrieve?

The tips the answer is grounded in -- if a claim isn't traceable to one of
these, the model made it up.

In [5]:
for d in coach.search(question):
    print(f"[{d['creator_id']}] {d['tip']}")

[skill-capped] Don't reflexively head back to bot with your ADC after a play — check where the next play is likely to happen first.
[mobalytics] Spam the 'on my way' ping repeatedly while roaming so your ally doesn't miss it.
[mobalytics] When sieging bot lane, place defensive Control Wards in the bushes behind your team to deny the enemy any teleport-play or flank setup.
[mobalytics] When pushing bot lane, place a Control Ward behind you so your jungler can set up a lane gank without being spotted.
[skill-capped] Learn when to break the standard roam-timing rules, which typically means giving up bot lane entirely.


## Compare the two prompt variants (P0-6)

Same retrieval, same question -- only the system prompt differs. This is the
A/B the P0-6 eval will judge at scale.

In [6]:
for label, instructions in [("BLUNT", COACH_BLUNT), ("WHY", COACH_WHY)]:
    variant = RAGBase(index=index, llm_client=client, instructions=instructions, model=model)
    print(f"### {label}\n")
    print(variant.rag(question))
    print("\n")

### BLUNT

1. **Before automatically returning to bot with your ADC, check the minimap and identify where the next play is happening.** If the fight is shifting mid or top, rotate there instead of defaulting back to lane. (skill-capped)

2. **Place a Control Ward in the bush behind your team when you're sieging bot lane** to deny enemy flanks and teleport plays while you're committed to the push. (mobalytics)

3. **Recognize when bot lane is no longer worth defending** — sometimes you need to break standard roaming rules and give up bot entirely to be where the win condition actually is. (skill-capped)


### WHY

Based on your issue of getting caught in bot lane, here are the concrete actions from my notes:

**1. Check where the next play is likely to happen before heading back to bot with your ADC**
Why: Automatically returning to bot can put you on the wrong side of the map right as the next fight breaks out elsewhere. (skill-capped)

This means after a play happens (kill, objective 